In [0]:
import importlib
from config.config import *

from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("delta").load(
    f"{bronze_path}/bronze_un_tourism_accommodation_hotels_raw"
)
display(df.head(10))

In [0]:
header_row = df.first()

new_columns = [
    str(header_row[c]).strip()
    for c in df.columns
]

df = df.filter(
    col(df.columns[0]) != header_row[df.columns[0]]
)

df = df.toDF(*new_columns)

df = df.drop(
    "indicator_previous_code", "notes"
)

In [0]:
df = df.withColumnRenamed(
    "reporter_area_label",
    "reporter_area"
).withColumnRenamed(
    "flag",
    "observation_flag"
)

In [0]:
df = df.withColumn(
    "reporter_area_code",
    expr("try_cast(reporter_area_code AS INT)")
).withColumn(
    "year",
    expr("try_cast(year AS INT)")
).withColumn(
    "hotel_capacity",
    expr("try_cast(value AS DOUBLE)")
)

df = df.drop("value")

In [0]:
df = df.withColumn(
    "geography_type",
    when(
        col("reporter_area").isNotNull(),
        "country"
    ).otherwise("unknown")
)

In [0]:
silver_df = df.select(
    "indicator_code",
    "indicator_label",
    "reporter_area_code",
    "reporter_area",
    "geography_type",
    "year",
    "hotel_capacity",
    "unit",
    "observation_flag",
    "flag_label",
)

display(silver_df)

In [0]:
output_path = f"{silver_path}/un_tourism_accommodation_hotels"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(output_path)

print("Data cleaning for un_tourism_accommodation_hotels completed")

In [0]:
spark.read.format("delta").load(output_path).display()